# Logits Agrgegation

In [1]:
def default_params(): 
    return {
        'current_model': 'M1',
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'name': 'blastwind/github-code-haskell-function',
            'content_column': 'full_solution', 
            'number_samples': 55,
        },
        'probs_bootstrapping_per_node':500,
        'default_max_position_embeddings' : 1024,
        'logits_path': '../data/raw_logits',
        'output_path': '../data/aggregated_nodes',
        'preprocessed_dataset_dir' : '../datax/functional_interpretability/dataset_preprocessing',
        'cache_dir': '../datax/hugging_face_cache',
        'log_file': '../datax/functional_interpretability/logit_extraction.log', 
        'callbacks_dir' : '../datax/functional_interpretability/callbacks',
        'causal_models': {
            'M1': 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2': 'codellama/CodeLlama-13b-hf', #https://huggingface.co/codellama/CodeLlama-13b-hf
            'M3': 'codellama/CodeLlama-34b-hf', # https://huggingface.co/codellama/CodeLlama-34b-hf
            'M4': 'codellama/CodeLlama-70b-hf', #https://huggingface.co/codellama/CodeLlama-70b-hf, 
            'M5': 'mistralai/Mistral-7B-v0.1', #https://huggingface.co/mistralai/Mistral-7B-v0.1
        },
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import random
import numpy as np
from statistics import mean, median
import os
import torch
import gc

In [3]:
from functional_interpretability_lib.loader import download_grammars
from functional_interpretability_lib.parser import create_parser

In [4]:
from datasets import load_dataset, Dataset

In [5]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, pipeline, Conversation, BitsAndBytesConfig, CodeLlamaTokenizer, LlamaForCausalLM, PreTrainedTokenizerFast, CodeLlamaTokenizerFast
from datasets import load_dataset

2024-04-05 20:21:41.114870: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-04-05 20:21:41.114930: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-04-05 20:21:41.116061: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-04-05 20:21:41.122710: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [6]:
import logging
#logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)
logging.basicConfig(
    filename=params['log_file'],
    filemode='a',
    format='%(asctime)s : %(levelname)s : %(message)s', 
    level=logging.INFO
    )

#### Haskell

In [7]:
# Function to recursively traverse the AST and find Monad definitions
def find_monad_definitions(node):
    monads = []
    def traverse_and_find_monad_definitions(node, monads:list):
        # Look for instance declarations
        if node.type == 'instance':
            for child in node.children:
                if child.type == 'instance_head':
                    if 'Monad' in child.text.decode('utf8'):
                        monads.append(child.parent)
                        break 
        # Recurse through children nodes
        for child in node.children:
            traverse_and_find_monad_definitions(child, monads)
    traverse_and_find_monad_definitions(node, monads)
    return monads


#### Model Loading

In [8]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     match params['quantization']:
               case 'int4':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
               case 'int8':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
               case 'float32':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
               case 'float16':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
               case _: 
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [9]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

#### Parsers

In [10]:
download_grammars(['haskell'])
parser, node_types, language = create_parser('haskell')

/usr/local/lib/python3.11/dist-packages/functional_interpretability_lib/grammars


#### Load Dataset

In [11]:
df_actual_ntp = pd.read_csv(params['logits_path'] + '/' + params['current_model'] + '_q_' + params['quantization'] + '/' + 'raw_logits.csv', index_col=0)
df_actual_ntp = df_actual_ntp.head()

In [12]:
df_actual_ntp.head(2)

,problem_type,problem,full_solution,entry_point,function,signature,context,unit_test_template,input_ids,max_prob,min_prob,actual_prob,loss
0,function,The function `calculateDiscount` takes a singl...,calculateDiscount :: Int -> Int\ncalculateDisc...,calculateDiscount,calculateDiscount amount\n | amount < 100 = a...,calculateDiscount :: Int -> Int,NaN,import Test.Hspec\n<FILL SIGNATURE>\n<FILL FUN...,"[8147, 4205, 2798, 4761, 3159, 1599, 3159, 13,...","[('<PRE>', 0.7492942214012146), ('_', 0.293801...","[('<s>', 6.322590020979568e-13), ('Gemeinsame'...","[('calculate', 4.767822403550781e-08), ('Dis',...",0.965619
1,function,The function `calculateThemeChange` takes two ...,calculateThemeChange :: String -> String -> St...,calculateThemeChange,calculateThemeChange current desired\n | curr...,calculateThemeChange :: String -> String -> St...,NaN,import Test.Hspec\n<FILL SIGNATURE>\n<FILL FUN...,"[8147, 17693, 7277, 4761, 1714, 1599, 1714, 15...","[('<PRE>', 0.7492938041687012), ('_', 0.293801...","[('<s>', 6.322610620820845e-13), ('Gemeinsame'...","[('calculate', 4.767828798435403e-08), ('Theme...",1.723406


#### Bootstrapping

In [13]:
#| export
def bootstrapping( np_data, np_func, size ):
    """Create a bootstrap sample given data and a function
    For instance, a bootstrap sample of means, or mediands. 
    The bootstrap replicates are a long as the original size
    we can choose any observation more than once (resampling with replacement:np.random.choice)
    """
    
    #Cleaning NaNs
    #np_data_clean = np_data[ np.logical_not( np.isnan(np_data) ) ] 
    
    #The size of the bootstrap replicate is as big as size
    #Creating the boostrap replicates as long as the orignal data size
    #This strategy might work as imputation 
    bootstrap_repl = [ np_func( np.random.choice( np_data, size=len(np_data) ) ) for i in range( size ) ]
    
    #logging.info("Covariate: " + cov) #Empirical Mean
    #logging.info("Empirical Mean: " + str(np.mean(np_data_clean))) #Empirical Mean
    #logging.info("Bootstrapped Mean: " + str( np.mean(bootstrap_repl) ) ) #Bootstrapped Mean
    
    return np.array( bootstrap_repl )

#### Token Binding

In [14]:
def convert_to_offset(
    point,              #point to convert
    lines: list         #list of lines in the source code
    ):
        """Convert the point to an offset"""
        row, column = point
        chars_in_rows = sum(map(len, lines[:row])) + row
        chars_in_columns = len(lines[row][:column])
        offset = chars_in_rows + chars_in_columns
        return offset

In [15]:
def bind_bpe_tokens(
    node,              #Tree sitter ast tree
    encoding,          #Token encoding
    actual_probs,      #Actual probabilities
    lines              #Source code Snippet
): 
    """Traverses the tree and bind the leaves with the corresponding node"""
    tree_node = {}
    tree_node['type'] = node.type
    tree_node['children'] = []
    tree_node['bindings'] = []

    node_span = [convert_to_offset(node.start_point, lines), convert_to_offset(node.end_point, lines)]
    for encoding_index, token_span in enumerate(encoding.offset_mapping):
        if (node_span[0] <= token_span[0] and token_span[0] < node_span[1]) \
        or (node_span[0] < token_span[1] and token_span[1] <= node_span[1]) \
        or (node_span[0] >= token_span[0] and token_span[1] >= node_span[1]) :
            tree_node['bindings'].append(actual_probs[encoding_index])
    
    for n in node.children:
        tree_node['children'].append(bind_bpe_tokens(n, encoding, actual_probs, lines))

    return tree_node
        

In [16]:
def process_bindings(
    node: dict,     #Binded AST tree with actual probabilities
) -> None:
    """process the bindings in the tree to agregate the probabilities"""
    node_actual_probs = [binding[1] for binding in node['bindings'] if isinstance(binding[1], float)]
    node['median_prob'] = node['max_prob'] = node['min_prob'] = node['avg_prob'] =  node['std'] = None
    ## len is zero if node correspond to FIRST_TOKEN = 'def' 
    if(len(node_actual_probs) > 0):
        ## BOOTSTRAPPING-> 
        node_actual_probs = bootstrapping(node_actual_probs, np.mean, size=500).tolist()
        ##
        node['median_prob'] = median(node_actual_probs) 
        node['max_prob'] = max(node_actual_probs) 
        node['min_prob'] = min(node_actual_probs)
        node['avg_prob'] = mean(node_actual_probs)
        node['std'] = np.std(node_actual_probs)
    for child in node['children']:
        process_bindings(child)

In [17]:
def compute_bindings_probs(df_actual_ntp, parser, tokenizer):
    binded_tree_col = []
    for index, row in df_actual_ntp.iterrows():
        tree = parser.parse(bytes(row[params['dataset']['content_column']], "utf8"))
        encoding = tokenizer(row[params['dataset']['content_column']], return_offsets_mapping=True, add_special_tokens=False)
        actual_logits = eval(row['actual_prob'])
        actual_logits.insert(0,(tokenizer.decode(eval(row['input_ids'])[0]),'FIRST_TOKEN'))
        binded_tree = bind_bpe_tokens(tree.root_node, encoding, actual_logits, row[params['dataset']['content_column']].split('\n'))
        binded_tree_col.append(binded_tree)
    df_actual_ntp['binded_tree'] = binded_tree_col
    df_actual_ntp['binded_tree'].apply(lambda binded_tree: process_bindings(binded_tree))

In [18]:
encoding = tokenizer(df_actual_ntp.iloc[0][params['dataset']['content_column']], return_offsets_mapping=True, add_special_tokens=False)
assert len(eval(df_actual_ntp.iloc[0]['input_ids'])) == len(encoding['input_ids'])

In [19]:
compute_bindings_probs(df_actual_ntp, parser, tokenizer)

In [20]:
df_actual_ntp.head(5)

,problem_type,problem,full_solution,entry_point,function,signature,context,unit_test_template,input_ids,max_prob,min_prob,actual_prob,loss,binded_tree
0,function,The function `calculateDiscount` takes a singl...,calculateDiscount :: Int -> Int\ncalculateDisc...,calculateDiscount,calculateDiscount amount\n | amount < 100 = a...,calculateDiscount :: Int -> Int,NaN,import Test.Hspec\n<FILL SIGNATURE>\n<FILL FUN...,"[8147, 4205, 2798, 4761, 3159, 1599, 3159, 13,...","[('<PRE>', 0.7492942214012146), ('_', 0.293801...","[('<s>', 6.322590020979568e-13), ('Gemeinsame'...","[('calculate', 4.767822403550781e-08), ('Dis',...",0.965619,"{'type': 'haskell', 'children': [{'type': 'sig..."
1,function,The function `calculateThemeChange` takes two ...,calculateThemeChange :: String -> String -> St...,calculateThemeChange,calculateThemeChange current desired\n | curr...,calculateThemeChange :: String -> String -> St...,NaN,import Test.Hspec\n<FILL SIGNATURE>\n<FILL FUN...,"[8147, 17693, 7277, 4761, 1714, 1599, 1714, 15...","[('<PRE>', 0.7492938041687012), ('_', 0.293801...","[('<s>', 6.322610620820845e-13), ('Gemeinsame'...","[('calculate', 4.767828798435403e-08), ('Theme...",1.723406,"{'type': 'haskell', 'children': [{'type': 'sig..."
2,function,The function `updatePath` takes two strings as...,import Data.List (isInfixOf)\n\nupdatePath :: ...,updatePath,updatePath existingPath newDir\n | newDir `is...,updatePath :: String -> String -> String,import Data.List (isInfixOf)\n\n,import Data.List (isInfixOf)\nimport Test.Hspe...,"[1053, 3630, 29889, 1293, 313, 275, 797, 5878,...","[('<PRE>', 0.7492926716804504), ('{', 0.305273...","[('<s>', 6.322408417115677e-13), ('<SU', 3.194...","[('import', 0.010562296956777573), ('Data', 0....",1.118250,"{'type': 'haskell', 'children': [{'type': 'imp..."
3,function,Design a function named `calculateDiscount` th...,calculateDiscount :: Float -> Int -> Bool -> F...,calculateDiscount,calculateDiscount price discountPercentage onC...,calculateDiscount :: Float -> Int -> Bool -> F...,NaN,import Test.Hspec\n<FILL SIGNATURE>\n<FILL FUN...,"[8147, 4205, 2798, 4761, 27842, 1599, 3159, 15...","[('<PRE>', 0.7492900490760803), ('_', 0.293801...","[('<s>', 6.321939499676077e-13), ('Gemeinsame'...","[('calculate', 4.7675776215783117e-08), ('Dis'...",0.853184,"{'type': 'haskell', 'children': [{'type': 'sig..."
4,function,The function `filterAndCount` takes a list of ...,import Data.List (filter)\n\nfilterAndCount ::...,filterAndCount,filterAndCount strs char threshold = length $ ...,filterAndCount :: [String] -> Char -> Int -> Int,import Data.List (filter)\n\n,import Data.List (filter)\nimport Test.Hspec\n...,"[1053, 3630, 29889, 1293, 313, 4572, 29897, 13...","[('<PRE>', 0.749289870262146), ('{', 0.3052741...","[('<s>', 6.321889626376143e-13), ('<SU', 3.194...","[('import', 0.010562398470938206), ('Data', 0....",0.974512,"{'type': 'haskell', 'children': [{'type': 'imp..."


In [21]:
df_actual_ntp.iloc[0]['binded_tree']

{'type': 'haskell',
 'children': [{'type': 'signature',
   'children': [{'type': 'variable',
     'children': [],
     'bindings': [('calculate', 'FIRST_TOKEN'),
      ('calculate', 4.767822403550781e-08),
      ('Dis', 0.0007291393121704459)],
     'median_prob': 0.0003645934951972407,
     'max_prob': 0.0007291393121704459,
     'min_prob': 4.767822403550781e-08,
     'avg_prob': 0.0003711553199027584,
     'std': 0.00025509769161271976},
    {'type': '::',
     'children': [],
     'bindings': [('count', 0.4988924562931061)],
     'median_prob': 0.4988924562931061,
     'max_prob': 0.4988924562931061,
     'min_prob': 0.4988924562931061,
     'avg_prob': 0.4988924562931061,
     'std': 0.0},
    {'type': 'fun',
     'children': [{'type': 'type_name',
       'children': [{'type': 'type',
         'children': [],
         'bindings': [('::', 0.03033667989075184)],
         'median_prob': 0.03033667989075184,
         'max_prob': 0.03033667989075184,
         'min_prob': 0.030336679890

#### SAVE

In [22]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [23]:
#create_folder(params['output_path'] + '/' + params['current_model'] + '_q_' + params['quantization'])
df_actual_ntp.to_csv('aggregated_nodes.csv')

In [24]:
torch.cuda.empty_cache()
gc.collect()

20